# 04 — Before vs After Award Lift Analysis (Junior vs Senior)

**Goal:** For each award-winning author, look at a +/-3 year window around their award year and
measure the "lift" in publication output (and citations, with caveats) after the award vs before.
Then compare this lift between Junior and Senior award winners.

**Window definition:** `before` = [award_year-3, award_year-1], `after` = [award_year+1, award_year+3].
The award year itself is excluded from both windows to avoid the award paper itself dominating
either side.

**Why works-level data, not the author summary endpoint:** OpenAlex's author-level `counts_by_year`
(used in notebook 02) only reliably covers roughly the last ~10 years. Since award years span
2000-2018, using it here would silently drop pre-award data for two-thirds of the dataset. Instead
we pull each author's individual works via `/works?filter=author.id:X`, which carry an exact
`publication_year` regardless of age.

**Metrics:**
- `pub_lift` = mean publications/year in `after` window - mean publications/year in `before` window.
  This is fully robust (publication_year is always exact, no truncation issue).
- `citation_lift` = mean citations/year (to papers published in that window) after - before. NOTE:
  each work's `cited_by_count` is cumulative-to-today, not year-specific, so this measures "citations
  accrued so far to papers from that window" rather than a clean time-anchored citation rate. Treated
  as a secondary, caveated metric.


## Setup

In [7]:
import pandas as pd
import numpy as np
import requests
import time
from pathlib import Path
from scipy import stats

np.random.seed(42)
DATA_DIR = Path('../conf_data')
FIG_DIR = Path('../conf_figures')
FIG_DIR.mkdir(exist_ok=True)

OPENALEX_API_KEY = "LojPLyiN5Hly8F5xX6YbHu"
MAILTO = "sherotowshaw@gmail.com"  # replace with your real email
BASE_URL = "https://api.openalex.org"

WINDOW = 3  # years before/after, excluding award year itself

author_stage = pd.read_csv(DATA_DIR / '02_award_authors_with_career_stage.csv')
author_stage = author_stage[author_stage['career_stage'].notna()].copy()
author_level = author_stage.drop_duplicates(subset='author_id')[
    ['author_id', 'author_name', 'career_stage', 'award_year', 'first_pub_year']
].copy()
print(author_level.shape)
author_level['career_stage'].value_counts()


(2701, 5)


career_stage
Senior    1941
Junior     760
Name: count, dtype: int64

**Note:** authors with multiple award papers appear once in `02_award_authors_with_career_stage.csv`
per paper. Here we take their FIRST award year as the event date for the before/after window, since
an author's career stage was computed relative to that award.


In [8]:
first_award = author_stage.groupby('author_id')['award_year'].min().rename('first_award_year')
author_level = author_level.drop(columns='award_year').merge(first_award, on='author_id', how='left')
author_level.head()


,author_id,author_name,career_stage,first_pub_year,first_award_year
0,https://openalex.org/A5026521600,Chenjun Xiao,Senior,2012.0,2018
1,https://openalex.org/A5014823249,Jincheng Mei,Junior,2014.0,2018
2,https://openalex.org/A5010677076,Martin Müller,Senior,1966.0,2018
3,https://openalex.org/A5054978902,John Hale,Senior,1930.0,2018
4,https://openalex.org/A5111222692,Chris Dyer,Senior,1970.0,2018


## Step 1 — Pull all works per author from OpenAlex

In [9]:
import pickle

WORKS_CACHE_PATH = DATA_DIR / '04_all_works_cache.pkl'

def fetch_author_works(author_id_short, per_page=200):
    works = []
    cursor = '*'
    while True:
        params = {
            'filter': f'author.id:{author_id_short}',
            'per-page': per_page,
            'cursor': cursor,
            'select': 'id,publication_year,cited_by_count',
            'api_key': OPENALEX_API_KEY,
            'mailto': MAILTO,
        }
        try:
            r = requests.get(f"{BASE_URL}/works", params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
        except Exception:
            break
        results = data.get('results', [])
        works.extend(results)
        cursor = data.get('meta', {}).get('next_cursor')
        if not cursor or not results:
            break
    return works


if WORKS_CACHE_PATH.exists():
    with open(WORKS_CACHE_PATH, 'rb') as f:
        cache = pickle.load(f)
    all_works = cache['all_works']
    failed_authors = cache['failed_authors']
    print(f"Loaded cached works for {len(all_works)} authors, {len(failed_authors)} failures (skipped fetch)")
else:
    all_works = {}
    failed_authors = []
    for i, aid in enumerate(author_level['author_id'].tolist()):
        aid_short = aid.split('/')[-1]
        try:
            all_works[aid] = fetch_author_works(aid_short)
        except Exception as e:
            failed_authors.append((aid, str(e)))
        time.sleep(0.05)

    with open(WORKS_CACHE_PATH, 'wb') as f:
        pickle.dump({'all_works': all_works, 'failed_authors': failed_authors}, f)
    print(f"Fetched works for {len(all_works)} authors, {len(failed_authors)} failures (cached to disk)")

print(f"Total works fetched: {sum(len(v) for v in all_works.values())}")

Fetched works for 2701 authors, 0 failures (cached to disk)
Total works fetched: 432047


## Step 2 — Compute per-author before/after windows

In [10]:
records = []
for _, row in author_level.iterrows():
    aid = row['author_id']
    award_year = row['first_award_year']
    works = all_works.get(aid, [])
    if not works:
        continue

    before_lo, before_hi = award_year - WINDOW, award_year - 1
    after_lo, after_hi = award_year + 1, award_year + WINDOW

    before_works = [w for w in works if w.get('publication_year') and before_lo <= w['publication_year'] <= before_hi]
    after_works = [w for w in works if w.get('publication_year') and after_lo <= w['publication_year'] <= after_hi]

    pubs_before = len(before_works)
    pubs_after = len(after_works)
    cites_before = sum(w.get('cited_by_count', 0) for w in before_works)
    cites_after = sum(w.get('cited_by_count', 0) for w in after_works)

    records.append({
        'author_id': aid,
        'author_name': row['author_name'],
        'career_stage': row['career_stage'],
        'award_year': award_year,
        'first_pub_year': row['first_pub_year'],
        'pubs_before': pubs_before,
        'pubs_after': pubs_after,
        'pubs_per_year_before': pubs_before / WINDOW,
        'pubs_per_year_after': pubs_after / WINDOW,
        'pub_lift': (pubs_after - pubs_before) / WINDOW,
        'cites_before': cites_before,
        'cites_after': cites_after,
        'cites_per_year_before': cites_before / WINDOW,
        'cites_per_year_after': cites_after / WINDOW,
        'citation_lift': (cites_after - cites_before) / WINDOW,
    })

lift_df = pd.DataFrame(records)
print(lift_df.shape)
lift_df.to_csv(DATA_DIR / '04_author_before_after_lift.csv', index=False)
lift_df.head()


(2700, 15)


,author_id,author_name,career_stage,award_year,first_pub_year,pubs_before,pubs_after,pubs_per_year_before,pubs_per_year_after,pub_lift,cites_before,cites_after,cites_per_year_before,cites_per_year_after,citation_lift
0,https://openalex.org/A5026521600,Chenjun Xiao,Senior,2018,2012.0,6,11,2.000000,3.666667,1.666667,14,156,4.666667,52.000000,47.333333
1,https://openalex.org/A5014823249,Jincheng Mei,Junior,2018,2014.0,5,13,1.666667,4.333333,2.666667,34,145,11.333333,48.333333,37.000000
2,https://openalex.org/A5010677076,Martin Müller,Senior,2018,1966.0,16,15,5.333333,5.000000,-0.333333,93,116,31.000000,38.666667,7.666667
3,https://openalex.org/A5054978902,John Hale,Senior,2018,1930.0,7,18,2.333333,6.000000,3.666667,820,348,273.333333,116.000000,-157.333333
4,https://openalex.org/A5111222692,Chris Dyer,Senior,2018,1970.0,123,42,41.000000,14.000000,-27.000000,17969,1232,5989.666667,410.666667,-5579.000000


**Coverage check:** authors whose `first_pub_year` is within the `before` window (i.e., they hadn't
started publishing yet 3 years before the award) will have artificially low/zero `pubs_before`,
which isn't a data gap but a true early-career effect — flagged separately, not excluded, since it's
informative for Junior authors specifically.


In [11]:
lift_df['before_window_start'] = lift_df['award_year'] - WINDOW
lift_df['career_started_during_before_window'] = lift_df['first_pub_year'] > lift_df['before_window_start']
print(lift_df.groupby('career_stage')['career_started_during_before_window'].mean())


career_stage
Junior    0.553947
Senior    0.005155
Name: career_started_during_before_window, dtype: float64


## Step 3 — Bootstrap CIs for lift, by career stage

In [12]:
def bootstrap_ci(data, n_boot=5000, ci=95):
    data = pd.Series(data).dropna().values
    if len(data) == 0:
        return np.nan, np.nan, np.nan
    boot_means = np.array([np.mean(np.random.choice(data, len(data), replace=True)) for _ in range(n_boot)])
    lo = np.percentile(boot_means, (100-ci)/2)
    hi = np.percentile(boot_means, 100 - (100-ci)/2)
    return np.mean(data), lo, hi

lift_metrics = ['pub_lift', 'citation_lift']
summary_rows = []
for metric in lift_metrics:
    for stage in ['Junior', 'Senior']:
        subset = lift_df.loc[lift_df['career_stage']==stage, metric]
        mean, lo, hi = bootstrap_ci(subset)
        summary_rows.append({'metric': metric, 'career_stage': stage, 'n': subset.notna().sum(),
                              'mean_lift': mean, 'ci_lower': lo, 'ci_upper': hi, 'median_lift': subset.median()})

lift_summary = pd.DataFrame(summary_rows)
lift_summary.to_csv(DATA_DIR / '04_lift_summary.csv', index=False)
lift_summary


,metric,career_stage,n,mean_lift,ci_lower,ci_upper,median_lift
0,pub_lift,Junior,760,1.398246,1.209211,1.598257,0.666667
1,pub_lift,Senior,1940,1.667010,1.359785,1.976654,1.000000
2,citation_lift,Junior,760,96.586404,63.629572,132.019539,14.000000
3,citation_lift,Senior,1940,21.365636,-53.578436,91.077315,0.000000


## Step 4 — Significance tests

In [13]:
test_rows = []
for metric in lift_metrics:
    j = lift_df.loc[lift_df['career_stage']=='Junior', metric].dropna()
    s = lift_df.loc[lift_df['career_stage']=='Senior', metric].dropna()
    stat_between, p_between = stats.mannwhitneyu(j, s, alternative='two-sided')
    test_rows.append({'metric': metric, 'test': 'Junior vs Senior lift (Mann-Whitney)',
                       'n_junior': len(j), 'n_senior': len(s), 'stat': stat_between, 'p_value': p_between})

for stage in ['Junior', 'Senior']:
    sub = lift_df[lift_df['career_stage']==stage]
    stat_pub, p_pub = stats.wilcoxon(sub['pubs_per_year_before'], sub['pubs_per_year_after'])
    test_rows.append({'metric': 'pubs_per_year', 'test': f'{stage} before vs after (Wilcoxon)',
                       'n_junior': None, 'n_senior': None, 'stat': stat_pub, 'p_value': p_pub})
    stat_cite, p_cite = stats.wilcoxon(sub['cites_per_year_before'], sub['cites_per_year_after'])
    test_rows.append({'metric': 'cites_per_year', 'test': f'{stage} before vs after (Wilcoxon)',
                       'n_junior': None, 'n_senior': None, 'stat': stat_cite, 'p_value': p_cite})

test_df = pd.DataFrame(test_rows)
test_df.to_csv(DATA_DIR / '04_lift_significance_tests.csv', index=False)
test_df


,metric,test,n_junior,n_senior,stat,p_value
0,pub_lift,Junior vs Senior lift (Mann-Whitney),760.0,1940.0,756406.0,2.914482e-01
1,citation_lift,Junior vs Senior lift (Mann-Whitney),760.0,1940.0,838900.0,2.366759e-08
2,pubs_per_year,Junior before vs after (Wilcoxon),NaN,NaN,35436.5,4.281063e-49
3,cites_per_year,Junior before vs after (Wilcoxon),NaN,NaN,64616.0,4.453856e-23
4,pubs_per_year,Senior before vs after (Wilcoxon),NaN,NaN,576750.5,5.621288e-35
5,cites_per_year,Senior before vs after (Wilcoxon),NaN,NaN,897639.0,3.531889e-01


In [14]:
traj_records = []
for _, row in author_level.iterrows():
    aid, stage, ay = row['author_id'], row['career_stage'], row['first_award_year']
    for w in all_works.get(aid, []):
        py = w.get('publication_year')
        if py is None:
            continue
        rel_year = py - ay
        if -5 <= rel_year <= 5:
            traj_records.append({
                'author_id': aid, 'treatment': 1 if stage == 'Junior' else 0,
                'rel_year': rel_year, 'works': 1, 'citations': w.get('cited_by_count', 0)
            })

long_df = pd.DataFrame(traj_records)
long_df = long_df.groupby(['author_id','treatment','rel_year'], as_index=False).agg(
    works=('works','sum'), citations=('citations','sum')
)
long_df.to_csv(DATA_DIR / '04_trajectory_long.csv', index=False)
print("Saved:", long_df.shape)

Saved: (23721, 5)
